In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
from datetime import datetime

start_time = datetime.now()

In [0]:
def audit_log(run_id, schema_name, table_name, start_time, status, row_count, error):
    end_time = datetime.now()
    log = {
        'run_id': run_id,
        'schema_name': schema_name,
        'table_name': table_name,
        'start_time': str(start_time),
        'end_time': str(end_time),
        'duration': (end_time - start_time).total_seconds(),
        'status': status,
        'rows_written': int(row_count or 0),
        'error_message': error
    }
    schema = ("run_id string, schema_name string, table_name string, "
              "start_time string, end_time string, duration double, "
              "status string, rows_written bigint, error_message string")

    spark.createDataFrame([log], schema) \
         .write.mode('append').saveAsTable('retail.ops.audit_logs')

def update_watermark(schema, table, latest_watermark):
    if latest_watermark is None:
        return
    spark.sql(f"""
        UPDATE retail.ops.watermark
        SET last_watermark = TIMESTAMP '{latest_watermark}'
        WHERE schema_name = '{schema}' AND table_name = '{table}'
    """)

In [0]:
dbutils.widgets.text('catalog', 'retail', 'Catalog')
dbutils.widgets.text('schema', 'silver', 'Schema')
dbutils.widgets.text('table', 'orders', 'Table')
dbutils.widgets.text('run_id', 'test', 'Run ID')
dbutils.widgets.text('watermark_column', '_ingested_at', 'Watermark Column')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table = dbutils.widgets.get('table')
run_id = dbutils.widgets.get('run_id')
watermark_column = dbutils.widgets.get('watermark_column')

In [0]:
latest_watermark= spark.sql(f'''
          SELECT MAX(last_watermark) FROM {catalog}.ops.watermark
          WHERE schema_name = '{schema}' AND table_name = '{table}'
          ''').first()[0]

In [0]:
latest_watermark

In [0]:
src_table_name = f'{catalog}.bronze.{table}'
bronze_batch = (
    spark.read.table(src_table_name) 
    .filter(F.col('_ingested_at') > latest_watermark)
)

silver_watermark = bronze_batch.select(F.max('_ingested_at')).first()[0]

### Transformation

In [0]:
df = spark.table(src_table_name)
display(df.limit(10))

In [0]:
## transformation
orders_df = (
    bronze_batch    
    .withColumn('txn_id', F.trim('txn_id'))
    .withColumn('customer_id', F.initcap(F.trim('customer_id')))
    .withColumn('txn_ts',  F.initcap(F.trim('txn_ts')))
    .withColumn('amount', F.round('amount', 2))
    .withColumnRenamed('_ingested_at', '_bronze_ingested_at')
    .withColumnRenamed('run_id','_bronze_run_id')
    .withColumn('_silver_updated_at', F.current_timestamp())   
)

## Dedupe over customer_ids
w = Window.partitionBy('txn_id').orderBy(F.col('_bronze_ingested_at').desc(), F.col('file_modification_time').desc())
orders_df = (
    orders_df
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn')==1)
    .drop('rn', 'file_modification_time')
)

In [0]:
display(orders_df.limit(10))

### Data Quality Checks

In [0]:
dq_check_conditions = {
    'null transaction id': "txn_id IS NULL",
    'null customer id': "customer_id IS NULL",
    'null transaction timestamp': "txn_ts IS NULL",
    'null amount': "amount IS NULL",
    'invalid amount': "amount < 0"
}

if not orders_df.isEmpty():

    dq_checks = (
        orders_df.withColumn('dq_checks', F.array_compact(F.array(*[F.when(F.expr(cond), F.lit(name)) for name, cond in dq_check_conditions.items()])))
    )